# Picking the RBF Bandwidth Without a Grid Search

**Topic:** Kernel ridge regression &nbsp;·&nbsp; **Difficulty:** Medium

The RBF kernel has a bandwidth, and if you get it wrong by an order of magnitude
the model is either a constant or a lookup table. The usual answer is a grid
search. The fast answer is the median heuristic: set the bandwidth from the
median squared pairwise distance in the data and you land in the right region, within a
factor of two of the best grid value, from which a short local search finishes
the job.

Implement kernel ridge from the normal equations, and compare the two.

## The data (created in the next cell)

- `X_train` `(300, 3)`, `y_train`, and `X_test` `(200, 3)`, `y_test`.
- The target is a smooth nonlinear function of the inputs plus noise.
- `gammas`: a grid of candidate RBF bandwidths.

## Your task

With **NumPy only** (no sklearn):

1. Implement the RBF kernel `K(a, b) = exp(-gamma * ||a - b||^2)`.
2. Kernel ridge: `alpha = (K + lam I)^{-1} y` with `lam = 1e-3`. Predictions on
   new points are `K(X_new, X_train) @ alpha`. Centre `y` by its training mean
   and add it back to predictions.
3. `gamma_median`, the median heuristic: `1 / (2 * median_sqdist)` where
   `median_sqdist` is the median of the **squared** pairwise distances among the
   training points (all pairs, including `i == j`).
4. Set `answers` to a dict with `"gamma_median"`, `"rmse_median"` (test RMSE at
   that bandwidth), `"grid_rmse"` (test RMSE for each gamma in `gammas`), and
   `"best_grid_rmse"`.

## Evaluation

Compared to the reference to 1e-6. The evaluation cell checks the median
heuristic lands within a factor of two of the best grid RMSE.

In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    Z = r.uniform(-2, 2, (n, 3))
    t = np.sin(2.0 * Z[:, 0]) * np.cos(1.3 * Z[:, 1]) + 0.4 * Z[:, 2] ** 2
    return Z, t + r.normal(0, 0.12, n)

X_train, y_train = make(300, 51)
X_test, y_test = make(200, 52)
gammas = np.logspace(-2, 1.5, 20)
lam = 1e-3
print(f"train {X_train.shape}, test {X_test.shape}, {len(gammas)} candidate gammas")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("gamma_median", [], 0.06829694579911105, -0.011435505022496602, 1e-06),
    ("rmse_median", [], 0.24943602052147912, -0.04176507209934514, 1e-06),
    ("grid_rmse", [20], 6.921993598520345, -0.5871135408660159, 1e-06),
    ("best_grid_rmse", [], 0.14361413020956257, -0.02404646486160015, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "rmse_median" in answers and "best_grid_rmse" in answers:
    _ok2 = answers["rmse_median"] <= 2.0 * answers["best_grid_rmse"]
    print(f"  {'median heuristic close':<24} {'ok' if _ok2 else 'wrong'}  "
          f"({answers['rmse_median']:.4f} vs best {answers['best_grid_rmse']:.4f})")
    _ok = _ok and _ok2

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The median heuristic works because it puts the typical pairwise distance right in
the kernel's sensitive region: much smaller and every point looks identical
(kernel ~ 1), much larger and every point is isolated (kernel ~ 0). It costs one distance matrix you already
computed, which is why it is the sensible starting point before spending compute
on a grid. Here it lands within a factor of two of the best grid value.

```python
import numpy as np

def sqdist(A, B):
    return (np.sum(A ** 2, 1)[:, None] + np.sum(B ** 2, 1)[None, :]
            - 2 * A @ B.T)

D_tr = np.maximum(sqdist(X_train, X_train), 0.0)
D_te = np.maximum(sqdist(X_test, X_train), 0.0)

my = y_train.mean()
yc = y_train - my
n = len(y_train)

def fit_predict(gamma):
    K = np.exp(-gamma * D_tr)
    a = np.linalg.solve(K + lam * np.eye(n), yc)
    return np.exp(-gamma * D_te) @ a + my

def rmse(p):
    return float(np.sqrt(np.mean((p - y_test) ** 2)))

gamma_median = float(1.0 / (2.0 * np.median(D_tr)))
rmse_median = rmse(fit_predict(gamma_median))

grid_rmse = np.array([rmse(fit_predict(g)) for g in gammas])

answers = {
    "gamma_median": gamma_median,
    "rmse_median": rmse_median,
    "grid_rmse": grid_rmse,
    "best_grid_rmse": float(grid_rmse.min()),
}
```